# PatchTST CI vs CD -- ETTh1 pred_len=96

Trains channel-independent (CI) and channel-dependent (CD) variants of PatchTST at
pred_len=96 on ETTh1 with identical hyperparameters and seed. Results are saved to
`results/ci_cd_ettch1.csv` with a `mode` column.

CI mode: each variate processed independently; encoder sees N patch tokens per channel.
CD mode: patches from all C variates concatenated before the encoder; attention runs
over C*N tokens simultaneously, allowing cross-variate context.

Expected finding: CI outperforms CD on ETTh1 (7 variates, strong local temporal
structure). With few variates, cross-variate attention introduces more noise than
signal, and the shared-weight CI design acts as implicit regularisation.

In [ ]:
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Dataset

In [ ]:
class ETTh1Dataset(Dataset):
    """ETTh1 multivariate dataset with chronological train/val/test split.

    Split: 8640 train / 2880 val / 2880 test rows.
    Normalization: z-score per channel, fit on train split only.
    """

    SPLIT_SIZES = {"train": 8640, "val": 2880, "test": 2880}
    TARGET_COLS = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]

    def __init__(self, csv_path: str, split: str, seq_len: int, pred_len: int) -> None:
        if split not in self.SPLIT_SIZES:
            raise ValueError(f"split must be one of {list(self.SPLIT_SIZES)}, got '{split}'.")

        df = pd.read_csv(csv_path)[self.TARGET_COLS].values.astype(np.float32)

        train_end = self.SPLIT_SIZES["train"]
        val_end = train_end + self.SPLIT_SIZES["val"]

        train_data = df[:train_end]
        self._mean = train_data.mean(axis=0)
        self._std = train_data.std(axis=0)
        self._std = np.where(self._std == 0, 1.0, self._std)

        data = (df - self._mean) / self._std

        if split == "train":
            self._data = data[:train_end]
        elif split == "val":
            self._data = data[train_end:val_end]
        else:
            self._data = data[val_end:]

        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self) -> int:
        return len(self._data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx: int):
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)

## Model

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, stride: int, d_model: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride = stride
        self.projection = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)
        self._d_model = d_model

    def _sinusoidal_pe(self, num_patches: int, device: torch.device) -> torch.Tensor:
        position = torch.arange(num_patches, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self._d_model, 2, device=device) * (-math.log(10000.0) / self._d_model))
        pe = torch.zeros(num_patches, self._d_model, device=device)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.squeeze(-1).unfold(dimension=-1, size=self.patch_size, step=self.stride)
        x = self.projection(x)
        pe = self._sinusoidal_pe(x.shape[1], x.device)
        return self.dropout(x + pe)


class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x


class TransformerEncoder(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)


class ForecastHead(nn.Module):
    def __init__(self, num_patches: int, d_model: int, pred_len: int, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(num_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(self.dropout(x.flatten(1)))


class PatchTST(nn.Module):
    """PatchTST with CI and CD mode support.

    channel_mixing=False (CI): each variate processed independently.
    channel_mixing=True  (CD): patches from all variates concatenated before encoder.
    """

    def __init__(
        self,
        seq_len,
        pred_len,
        num_variates,
        patch_size=16,
        stride=8,
        d_model=128,
        num_heads=16,
        num_layers=3,
        dropout=0.2,
        channel_mixing=False,
    ) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads}).")
        self.num_variates = num_variates
        self.channel_mixing = channel_mixing
        self.num_patches = (seq_len - patch_size) // stride + 1
        self.embedding = PatchEmbedding(patch_size=patch_size, stride=stride, d_model=d_model, dropout=dropout)
        self.encoder = TransformerEncoder(d_model=d_model, num_heads=num_heads, num_layers=num_layers, dropout=dropout)
        self.head = ForecastHead(num_patches=self.num_patches, d_model=d_model, pred_len=pred_len, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L, 1)
        x = self.embedding(x)  # (B*C, N, D)
        if self.channel_mixing:
            x = x.reshape(B, C * self.num_patches, -1)  # (B, C*N, D)
            x = self.encoder(x)  # (B, C*N, D)
            x = x.reshape(B * C, self.num_patches, -1)  # (B*C, N, D)
        else:
            x = self.encoder(x)  # (B*C, N, D)
        x = self.head(x)  # (B*C, pred_len)
        return x.reshape(B, C, -1).permute(0, 2, 1)  # (B, pred_len, C)

## Training Infrastructure

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 10, checkpoint_path: str = "best_model.pt") -> None:
        self.patience = patience
        self.checkpoint_path = checkpoint_path
        self.best_val_mse = float("inf")
        self.counter = 0
        self.best_epoch = 0

    def step(self, val_mse: float, model: nn.Module, epoch: int) -> bool:
        if val_mse < self.best_val_mse:
            self.best_val_mse = val_mse
            self.counter = 0
            self.best_epoch = epoch
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> tuple:
    mse = torch.mean((pred - target) ** 2).item()
    mae = torch.mean(torch.abs(pred - target)).item()
    return mse, mae


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        mse, mae = compute_metrics(pred.detach(), y)
        batch = x.size(0)
        total_mse += mse * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)
        mse, mae = compute_metrics(pred, y)
        batch = x.size(0)
        total_mse += mse * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n


def run_mode(mode: str, csv_path: str, config: dict, results_dir: Path, ckpt_dir: Path):
    """Train or load one CI/CD run and return result dict."""
    channel_mixing = mode == "CD"
    pred_len = config["pred_len"]
    ckpt_path = str(ckpt_dir / f"patchtst_cicd_{mode.lower()}_pred{pred_len}.pt")

    torch.manual_seed(config["seed"])
    random.seed(config["seed"])
    np.random.seed(config["seed"])

    train_ds = ETTh1Dataset(csv_path, "train", config["seq_len"], pred_len)
    val_ds = ETTh1Dataset(csv_path, "val", config["seq_len"], pred_len)
    test_ds = ETTh1Dataset(csv_path, "test", config["seq_len"], pred_len)

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = PatchTST(
        seq_len=config["seq_len"],
        pred_len=pred_len,
        num_variates=config["num_variates"],
        patch_size=config["patch_size"],
        stride=config["stride"],
        d_model=config["d_model"],
        num_heads=config["num_heads"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        channel_mixing=channel_mixing,
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())

    # If a CI checkpoint already exists from the main training run, load it directly.
    ci_ckpt = results_dir / "checkpoints" / f"patchtst_pred{pred_len}_best.pt"
    if mode == "CI" and ci_ckpt.exists():
        print(f"[{mode}] Loading existing checkpoint: {ci_ckpt}")
        model.load_state_dict(torch.load(ci_ckpt, map_location=DEVICE, weights_only=True))
        test_mse, test_mae = evaluate(model, test_loader)
        best_epoch = "loaded"
        best_val_mse = float("nan")
    else:
        print(f"[{mode}] Training from scratch | params: {total_params:,}")
        optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)
        # Linear warmup over first warmup_epochs, then cosine decay.
        warmup_epochs = config["warmup_epochs"]

        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return (epoch + 1) / warmup_epochs
            progress = (epoch - warmup_epochs) / max(1, config["epochs"] - warmup_epochs)
            return 0.5 * (1 + math.cos(math.pi * progress))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        criterion = nn.MSELoss()
        early_stopping = EarlyStopping(patience=config["patience"], checkpoint_path=ckpt_path)
        t0 = time.time()
        for epoch in range(1, config["epochs"] + 1):
            train_mse, _ = train_one_epoch(model, train_loader, optimizer, criterion)
            val_mse, _ = evaluate(model, val_loader)
            scheduler.step()
            if epoch % 5 == 0 or epoch == 1:
                lr_now = optimizer.param_groups[0]["lr"]
                print(
                    f'  [{mode}] Epoch {epoch:3d}/{config["epochs"]} | '
                    f"train MSE {train_mse:.4f} | val MSE {val_mse:.4f} | "
                    f"lr {lr_now:.2e} | {time.time()-t0:.0f}s"
                )
            if early_stopping.step(val_mse, model, epoch):
                print(f"  [{mode}] Early stop at epoch {epoch}. Best: epoch {early_stopping.best_epoch}.")
                break
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
        test_mse, test_mae = evaluate(model, test_loader)
        best_epoch = early_stopping.best_epoch
        best_val_mse = early_stopping.best_val_mse
        print(
            f"  [{mode}] Test MSE: {test_mse:.4f} | Test MAE: {test_mae:.4f} | "
            f"Best val MSE: {best_val_mse:.4f} @ epoch {best_epoch}"
        )

    return {
        "mode": mode,
        "pred_len": pred_len,
        "test_mse": round(test_mse, 6),
        "test_mae": round(test_mae, 6),
        "best_val_mse": round(best_val_mse, 6) if not math.isnan(best_val_mse) else None,
        "best_epoch": best_epoch,
        "num_params": total_params,
        "seed": config["seed"],
    }

## Config

In [ ]:
CSV_PATH = "/kaggle/input/datasets/alaaelmor/ettsmall/ETTh1.csv"
RESULTS_DIR = Path("results")
CKPT_DIR = Path("results/checkpoints")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# All hyperparameters are identical across CI and CD runs.
# This is the PatchTST/64 paper config for ETTh1.
CONFIG = {
    "pred_len": 96,
    "seq_len": 512,
    "num_variates": 7,
    "patch_size": 16,
    "stride": 8,
    "d_model": 128,
    "num_heads": 16,
    "num_layers": 3,
    "dropout": 0.2,
    "lr": 1e-4,
    "warmup_epochs": 10,
    "batch_size": 128,
    "epochs": 100,
    "patience": 10,
    "seed": 42,
}

## Run CI and CD

In [ ]:
results = []

for mode in ["CI", "CD"]:
    print(f'\n{"=" * 60}')
    print(f"Mode: {mode}")
    print(f'{"=" * 60}')
    result = run_mode(mode, CSV_PATH, CONFIG, RESULTS_DIR, CKPT_DIR)
    results.append(result)

results_df = pd.DataFrame(results)
results_path = RESULTS_DIR / "ci_cd_ettch1.csv"
results_df.to_csv(results_path, index=False)

print("\n=== CI vs CD Results ===")
print(results_df[["mode", "pred_len", "test_mse", "test_mae", "best_epoch"]].to_string(index=False))

ci_mse = results_df.loc[results_df["mode"] == "CI", "test_mse"].values[0]
cd_mse = results_df.loc[results_df["mode"] == "CD", "test_mse"].values[0]
ratio = cd_mse / ci_mse
winner = "CI" if ci_mse < cd_mse else "CD"
print(f"\nCD/CI MSE ratio: {ratio:.4f} -- {winner} wins.")
print(f"Results saved to {results_path}")

## Verify Output Files

In [ ]:
required = [RESULTS_DIR / "ci_cd_ettch1.csv"]

all_present = True
for path in required:
    exists = path.exists()
    print(f'  [{"OK" if exists else "MISSING"}] {path}')
    if not exists:
        all_present = False

if not all_present:
    raise RuntimeError("Output file missing. Do not close the session.")
print("\nAll output files verified.")